# KAIL — Notebook 03: SPIN Warm-up (Stage 1)

> **Kora AI Lab** | H-AZR Research Pipeline

## What this notebook does

**Stage 1 — SPIN (Self-Play Fine-Tuning)**

SPIN trains the model to distinguish its *current* outputs from a *previous version* of itself using a DPO-style objective. No RL, no reward model — just iterative self-distillation.

**Why SPIN before H-AZR?**
Our hypothesis: a better-aligned model (post-SPIN) should have *fewer active H-Neurons* at baseline, making Stage 3 more efficient. This notebook tests that hypothesis — run `02_h_neuron_probe.ipynb` again after this to compare.

**Prerequisites:** Run notebooks 00 and 01 first.

**Produces:** `spin_checkpoint/` → input for `04_h_azr_training.ipynb` (Scenario C)

**Runtime:** ~2h on free Colab T4

In [ ]:
# @title Install (skip if already done in 00_setup)
!pip install -q transformers==4.44.0 trl==0.9.6 peft==0.12.0 bitsandbytes==0.43.3 \
              accelerate==0.33.0 datasets==2.20.0

In [ ]:
# @title Imports & config
import torch
import os
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import DPOConfig, DPOTrainer
from datasets import Dataset

MODEL_NAME = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
SPIN_ITERATIONS = 3          # T iterations of self-play
SAMPLES_PER_ITER = 256       # dataset size per iteration
CHECKPOINT_DIR = "/content/drive/MyDrive/KAIL/checkpoints"  # from 00_setup
SPIN_OUTPUT = os.path.join(CHECKPOINT_DIR, "spin_final")

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"SPIN iterations: {SPIN_ITERATIONS}")
print(f"Output: {SPIN_OUTPUT}")

In [ ]:
# @title Load base model in 4-bit
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print(f"VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
# @title SPIN dataset builder
# SPIN core idea: generate responses from current model ("chosen" if better)
# vs previous model ("rejected"), then train to prefer current over previous.
# We bootstrap with human-written prompts and use the model itself to generate.

SEED_PROMPTS = [
    "Explain what a Python decorator is and write a simple example.",
    "Write a function that implements quicksort in Python.",
    "What is the difference between a list and a tuple in Python?",
    "Write a class that implements a binary search tree in Python.",
    "Explain Big O notation with examples.",
    "Write a Python function that parses a JSON file and returns a dict.",
    "What is recursion? Show a recursive implementation of merge sort.",
    "Write a Python generator that yields Fibonacci numbers.",
    "Explain the difference between shallow copy and deep copy in Python.",
    "Write a Python function that validates an email address using regex.",
    "What are Python context managers? Write a custom one.",
    "Implement a LRU cache in Python without using functools.",
    "Write a function that finds all permutations of a string.",
    "Explain Python's GIL and when it matters.",
    "Write a function that rotates a matrix 90 degrees clockwise.",
    "Implement a simple rate limiter in Python.",
]


def generate_response(prompt: str, max_tokens: int = 400) -> str:
    """Generate a response from the current model."""
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512).to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            do_sample=True,
            temperature=0.8,
            top_p=0.95,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)


def build_spin_dataset(prompts, prev_responses=None, n_samples=None):
    """
    Build SPIN dataset for one iteration.
    - chosen: current model response (sampled now)
    - rejected: previous model response (from last iteration, or greedy baseline)
    """
    if n_samples:
        prompts = (prompts * (n_samples // len(prompts) + 1))[:n_samples]

    print(f"Generating responses for {len(prompts)} prompts...")
    chosen = [generate_response(p) for p in prompts]

    if prev_responses is None:
        # First iteration: use greedy (lower temperature) as "rejected"
        rejected = [generate_response(p) for p in prompts]  # different sample
    else:
        rejected = prev_responses

    return Dataset.from_dict({
        "prompt": prompts,
        "chosen": chosen,
        "rejected": rejected,
    })

print("Dataset builder ready.")

In [ ]:
# @title SPIN Training Loop
# SPIN runs T iterations. Each iteration:
#   1. Generate current model responses (chosen)
#   2. Use previous iteration responses as rejected
#   3. Train with DPO objective

prev_responses = None
prompts_extended = (SEED_PROMPTS * (SAMPLES_PER_ITER // len(SEED_PROMPTS) + 1))[:SAMPLES_PER_ITER]

for iteration in range(SPIN_ITERATIONS):
    print(f"\n{'='*50}")
    print(f"SPIN Iteration {iteration + 1} / {SPIN_ITERATIONS}")
    print(f"{'='*50}")

    # Build dataset
    spin_dataset = build_spin_dataset(
        prompts_extended,
        prev_responses=prev_responses,
        n_samples=SAMPLES_PER_ITER,
    )

    # Save current responses to use as "rejected" in next iteration
    prev_responses = spin_dataset["chosen"]

    # DPO training config
    dpo_config = DPOConfig(
        output_dir=os.path.join(CHECKPOINT_DIR, f"spin_iter_{iteration+1}"),
        num_train_epochs=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        learning_rate=5e-5,
        warmup_ratio=0.1,
        lr_scheduler_type="cosine",
        fp16=True,
        beta=0.1,  # DPO beta
        max_length=512,
        max_prompt_length=256,
        logging_steps=10,
        save_steps=50,
        report_to="none",
        remove_unused_columns=False,
    )

    trainer = DPOTrainer(
        model=model,
        args=dpo_config,
        train_dataset=spin_dataset,
        processing_class=tokenizer,
    )

    trainer.train()
    print(f"Iteration {iteration+1} complete.")

# Save final SPIN model
model.save_pretrained(SPIN_OUTPUT)
tokenizer.save_pretrained(SPIN_OUTPUT)
print(f"\nSPIN complete. Final model saved to: {SPIN_OUTPUT}")

In [ ]:
# @title Quick comparison: before vs after SPIN
TEST_PROMPT = "What is the time complexity of Python's list.sort()? Explain why."

response = generate_response(TEST_PROMPT)
print("Post-SPIN response:")
print("-" * 50)
print(response)
print("-" * 50)
print("\nNext steps:")
print("1. Re-run 02_h_neuron_probe.ipynb on this checkpoint")
print("   → Does SPIN reduce H-Neurons? That's Scenario B data.")
print("2. Use this checkpoint as input for 04_h_azr_training.ipynb (Scenario C)")